# Simon's Algorithm
**Notebook:** Finds the hidden period string $b$ of a 2-to-1 function using quantum parallelism

## Overview

**Simon's problem:** Given a function $f:\{0,1\}^n \to \{0,1\}^n$ promised to satisfy
$f(x) = f(y) \iff x \oplus y \in \{0^n, b\}$ for some hidden string $b$, find $b$.

- **Classical complexity:** $O(2^{n/2})$ queries (birthday paradox)
- **Quantum complexity:** $O(n)$ queries â€” **exponential speedup**

### Algorithm
1. Start with $|0\rangle^n|0\rangle^n$
2. Apply Hadamard to the first register â†’ uniform superposition
3. Query the oracle: $|x\rangle|0\rangle \to |x\rangle|f(x)\rangle$
4. Apply Hadamard to the first register again
5. **Measure first register** â†’ get $z$ such that $z \cdot b = 0 \pmod{2}$
6. Repeat $n-1$ times to get a system of equations, solve for $b$

### Post-processing
Each measurement gives a vector $z$ satisfying $z \cdot b = 0 \pmod 2$.
After $n-1$ linearly independent measurements, $b$ can be recovered by Gaussian elimination.

### Oracle used here
The oracle encodes the period via CNOT gates:
- Copy $x$ into the output register
- XOR the output with $b$ when the first input bit is 1
This ensures $f(x) = f(x \oplus b)$.

In [ ]:
# â”€â”€ Imports â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import qiskit as qk
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from math import pi
import numpy as np

# qiskit_textbook provides a reference simon_oracle implementation.
# If not installed (pip install qiskit-textbook), the fallback below is used.
try:
    from qiskit_textbook.tools import simon_oracle
except ImportError:
    # â”€â”€ Fallback Simon oracle â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Constructs a quantum circuit that implements f(x) = f(x XOR b).
    # The circuit acts on 2n qubits: first n = input register, last n = output register.
    def simon_oracle(b):
        """Simon oracle: f(x) = f(x XOR b). Maps |xâŸ©|0âŸ© â†’ |xâŸ©|f(x)âŸ©.

        Oracle construction:
          1. Copy step â€” CNOT from each input qubit i to output qubit n+i:
               |xâŸ©|0âŸ© â†’ |xâŸ©|xâŸ©
             This ensures f(x) includes x as a base.
          2. XOR-with-b step â€” for each bit position i where b[i]='1',
             apply CNOT from input qubit 0 to output qubit n+i:
               when x[0]=1, output register gets XORed with b
             This encodes the 2-to-1 property: f(x) = f(x XOR b).
        """
        n = len(b)
        qc = QuantumCircuit(2 * n)

        # Step 1: copy input register into output register
        for i in range(n):
            qc.cx(i, n + i)   # CNOT: control=x[i], target=output[i] â†’ output[i] ^= x[i]

        # Step 2: XOR output with b, controlled on the first input qubit (x[0])
        # When x[0]=1, this flips output bits where b[i]='1',
        # guaranteeing f(x) = f(x XOR b) for the promised period b.
        for i, bit in enumerate(b):
            if bit == '1':
                qc.cx(0, n + i)   # CNOT: control=x[0], target=output[i] â†’ encodes period b
        return qc

In [ ]:
# â”€â”€ Build the Simon circuit for b = '11' â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# b = '11' means the hidden period string is "11" (both bits set).
# The function f satisfies: f(x) = f(x XOR 11) for all x.
# n = 2 input qubits â†’ circuit needs 2*n = 4 qubits total (input + output registers).
b = '11'
n = len(b)                              # n=2: number of input/output qubits each
simon_circuit_2 = QuantumCircuit(n*2, n)  # 4 qubits, 2 classical bits (measure input reg only)

# Step 1: Hadamard on input register (qubits 0..n-1)
# Creates equal superposition: |0âŸ©^n â†’ (1/âˆš2^n) Î£_x |xâŸ©
# All 2^n = 4 basis states are queried simultaneously in the next step.
simon_circuit_2.h(range(n))

# Step 2: Apply the Simon oracle
# |xâŸ©|0âŸ© â†’ |xâŸ©|f(x)âŸ©
# After this, measuring the output register would collapse the input to
# a superposition of {x, x XOR b} â€” the two inputs that share the same output.
simon_circuit_2 = simon_circuit_2.compose(simon_oracle(b))

# Step 3: Hadamard on input register again
# This second H layer is the key quantum step: it creates interference so that
# only states z satisfying z Â· b = 0 (mod 2) appear in the measurement.
# (The output register is traced out / ignored after measurement.)
simon_circuit_2.h(range(n))

# Step 4: Measure input register into classical bits
# Each shot yields a vector z where z Â· b = 0 (mod 2).
# Repeating gives n-1 linearly independent equations to solve for b.
simon_circuit_2.measure(range(n), range(n))
simon_circuit_2.draw(output='mpl', justify='none')

In [ ]:
# â”€â”€ Simulate the circuit â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Run 1024 shots on the local AerSimulator.
# Each shot returns a 2-bit string z satisfying z Â· b = 0 (mod 2).
# For b='11', the valid z values are those where z[0] XOR z[1] = 0,
# i.e., z âˆˆ {'00', '11'} â€” only these should appear in the histogram.
b = '11'
n = len(b)
backend_sim = AerSimulator()
shots = 1024
job = backend_sim.run(transpile(simon_circuit_2, backend_sim), shots=shots)
sim_counts = job.result().get_counts()
print("Simulator results:")
plot_histogram(sim_counts)

In [ ]:
# â”€â”€ Post-processing: verify z Â· b = 0 (mod 2) for every measurement â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Simon's algorithm guarantees that every measured z satisfies z Â· b = 0 (mod 2).
# This inner-product condition is verified here for each distinct outcome.
#
# z Â· b (mod 2) = (z[0]*b[0] + z[1]*b[1] + ...) mod 2
# For b='11': z Â· b = z[0] + z[1] (mod 2) â€” must equal 0, so z[0] = z[1].
# Valid outputs: '00' (0Â·1 + 0Â·1 = 0) and '11' (1Â·1 + 1Â·1 = 2 â‰¡ 0).
print(f"Secret string b = '{b}'")
print(f"Measurement results (z Â· b must = 0 mod 2):\n")

b_vec = [int(c) for c in b]   # convert b string to list of ints, e.g. '11' â†’ [1,1]
all_ok = True
for z, count in sorted(sim_counts.items(), key=lambda x: -x[1]):
    z_vec = [int(c) for c in z]   # convert measurement string to int list
    # Compute the binary dot product z Â· b mod 2
    dot = sum(zi * bi for zi, bi in zip(z_vec, b_vec)) % 2
    ok = 'âœ“' if dot == 0 else 'âœ—'   # mark valid (dot=0) or invalid (dot=1)
    if dot != 0:
        all_ok = False               # flag any violation of the zÂ·b=0 guarantee
    print(f"  z={z}  count={count:4d}  zÂ·b={dot} {ok}")

# Summary: if all_ok is True, every measurement is consistent with Simon's guarantee.
# In practice these equations are fed into Gaussian elimination to recover b.
print(f"\n{'All measurements satisfy zÂ·b=0 âœ“' if all_ok else 'Some measurements violate zÂ·b=0 âœ—'}")